# Phase 5: LIME Local Explanation

Perturbation-based **local explanation** for both image and text modalities.
Treats the entire model as a black box — independently validates findings
from Grad-CAM (Phase 2), Attention (Phase 3), and SHAP (Phase 4).

| Component | Configuration |
|---|---|
| Image LIME | Superpixel perturbation, 1000 samples |
| Text LIME | Word removal, 500 samples, Vietnamese syllable splitting |
| Output | Sigmoid pseudo-classification [low, high] |

**Priority:** Lowest among 4 XAI methods. Can be moved to thesis appendix if needed.

---
### STEP 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### STEP 2: Clone and install

In [ ]:
!rm -rf /content/SE365
!git clone -b xai-v3 https://github.com/lechihoang/SE365.git /content/SE365
%cd /content/SE365
!pip install -q -r requirements.txt
!pip install -q lime scikit-image

### STEP 3: Download and extract data

In [ ]:
!rm -rf ./data
!cp /content/drive/MyDrive/SE365/data.zip ./data.zip
!unzip -q data.zip
!rm data.zip
!ls -la ./data

### STEP 4: Configuration

In [ ]:
import os, sys, time, warnings
warnings.filterwarnings('ignore')

DRIVE_ROOT   = '/content/drive/MyDrive/SE365'
PROJECT_ROOT = '/content/SE365'
EXP_ID       = 'EXP_060A_bestsequential_full_configuration'

EXP_DIR      = f'{DRIVE_ROOT}/experiments/{EXP_ID}'
XAI_OUT_DIR  = f'{EXP_DIR}/xai/lime'
DATA_DIR     = f'{PROJECT_ROOT}/data/text'
IMAGE_DIR    = f'{PROJECT_ROOT}/data/image'

NUM_LIME_SAMPLES   = 5
NUM_IMAGE_PERTURBS = 1000
NUM_TEXT_PERTURBS  = 500

os.makedirs(XAI_OUT_DIR, exist_ok=True)
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f'PROJECT_ROOT       : {PROJECT_ROOT}')
print(f'EXP_DIR            : {EXP_DIR}')
print(f'XAI_OUT_DIR        : {XAI_OUT_DIR}')
print(f'NUM_LIME_SAMPLES   : {NUM_LIME_SAMPLES}')
print(f'NUM_IMAGE_PERTURBS : {NUM_IMAGE_PERTURBS}')
print(f'NUM_TEXT_PERTURBS  : {NUM_TEXT_PERTURBS}')

### STEP 5: Imports and Seed

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 5 — Step 5 — Imports and Seed')
print('='*60)

import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image as PILImage

from xai.config import (
    TARGET_NAMES, FACTOR_NAMES, DISPLAY_NAMES, NUM_TARGETS,
    DEFAULT_SEED, DEFAULT_DPI,
    BEST_TEXT_MODEL, BEST_IMAGE_MODEL,
)
from xai.utils import (
    get_device, set_seed, get_tokenizer, get_image_processor,
    load_model, load_single_sample, get_prediction,
    save_raw_values, get_metadata,
)
from xai.lime_explainer import (
    LIMEExplainer,
    run_lime_image, run_lime_text,
    save_lime_image_explanation, save_lime_text_explanation,
)

SEED = DEFAULT_SEED
set_seed(SEED)
device = get_device()

print(f'Device : {device}')
print(f'Seed   : {SEED}')
print(f'Done ({time.time()-t0:.1f}s)')

### STEP 6: Load Model

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 5 — Step 6 — Load Model')
print('='*60)

model, config = load_model(EXP_DIR, device=device)
print(f'Model : {model.__class__.__name__} ({sum(p.numel() for p in model.parameters()):,} params)')
print(f'Done ({time.time()-t0:.1f}s)')

### STEP 7: Tokenizer, Image Processor, Sample Selection

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 5 — Step 7 — Tokenizer & Sample Selection')
print('='*60)

text_model_name = config.get('text_model_name', BEST_TEXT_MODEL)
image_model_name = config.get('image_model_name', BEST_IMAGE_MODEL)
tokenizer = get_tokenizer(text_model_name)
image_processor = get_image_processor(image_model_name)

test_csv = os.path.join(DATA_DIR, 'test.csv')
val_csv  = os.path.join(DATA_DIR, 'val.csv')
SPLIT_CSV = test_csv if os.path.isfile(test_csv) else val_csv
SPLIT_NAME = 'test' if SPLIT_CSV == test_csv else 'validation'
df_split = pd.read_csv(SPLIT_CSV)

SAMPLE_INDICES = list(range(min(NUM_LIME_SAMPLES, len(df_split))))

print(f'Split    : {SPLIT_NAME} ({len(df_split)} samples)')
print(f'Selected : {SAMPLE_INDICES}')
print(f'Done ({time.time()-t0:.1f}s)')

### STEP 8: Generate Sample Manifest

Documents which samples are used and why — so you never have to reopen the dataset.

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 5 — Step 8 — Sample Manifest')
print('='*60)

manifest_rows = []
for sidx in SAMPLE_INDICES:
    row = df_split.iloc[sidx]
    s = load_single_sample(
        csv_path=SPLIT_CSV, idx=sidx,
        tokenizer=tokenizer, image_processor=image_processor,
        image_dir=IMAGE_DIR, device=device,
    )
    r = get_prediction(model, s)
    manifest_rows.append({
        'sample_idx': sidx,
        'text': s['text'][:200],
        'num_images': s['num_real_images'],
        'predictions': {k: round(v, 3) for k, v in r['predictions'].items()},
        'ground_truth': {k: round(v, 2) for k, v in r['ground_truth'].items()},
        'mean_mae': round(r['mean_mae'], 3),
    })

# Save manifest
manifest_json_path = os.path.join(XAI_OUT_DIR, 'selected_samples_manifest.json')
save_raw_values(manifest_rows, manifest_json_path)

# Print preview
for m in manifest_rows:
    print(f'\n--- Sample {m["sample_idx"]} ---')
    print(f'  Text (200ch): {m["text"]}...')
    print(f'  Images: {m["num_images"]}, MAE: {m["mean_mae"]}')

print(f'\nDone ({time.time()-t0:.1f}s)')

### STEP 9: Display First Sample (Preview Before LIME)

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 5 — Step 9 — Sample Preview')
print('='*60)

demo_idx = SAMPLE_INDICES[0]
sample = load_single_sample(
    csv_path=SPLIT_CSV, idx=demo_idx,
    tokenizer=tokenizer, image_processor=image_processor,
    image_dir=IMAGE_DIR, device=device,
)
result = get_prediction(model, sample)

print(f'Sample {demo_idx}')
print(f'Text: {sample["text"][:150]}...')
print(f'Images: {sample["num_real_images"]}')
print(f'\n{"Target":<28s} {"Pred":>8s} {"GT":>8s}')
print('-'*46)
for name in TARGET_NAMES:
    print(f'{name:<28s} {result["predictions"][name]:8.3f} {result["ground_truth"][name]:8.2f}')

# Display image
if sample['loaded_images']:
    fig, ax = plt.subplots(1, 1, figsize=(4, 4))
    ax.imshow(sample['loaded_images'][0])
    ax.set_title(f'Sample {demo_idx} — Image 0')
    ax.axis('off')
    plt.tight_layout()
    plt.show()

print(f'Done ({time.time()-t0:.1f}s)')

### STEP 10: Single-Target Text LIME Demo

Run LIME Text for `food_score` (target 0) on the demo sample.

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 5 — Step 10 — Text LIME Demo (food_score)')
print('='*60)

text_exp = run_lime_text(
    model=model, sample=sample, score_index=0,
    tokenizer=tokenizer, device=device,
    num_features=10, num_samples=NUM_TEXT_PERTURBS, seed=SEED,
)

print(f'\nTop 10 words by LIME importance (food_score):')
for word, weight in text_exp.as_list(label=1)[:10]:
    sign = '+' if weight > 0 else ''
    print(f'  {word:<20s} {sign}{weight:.4f}')

print(f'\nDone ({time.time()-t0:.1f}s)')

### STEP 11: Single-Target Image LIME Demo

Run LIME Image for `food_score` on the first image.

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 5 — Step 11 — Image LIME Demo (food_score)')
print('='*60)

image_exp = run_lime_image(
    model=model, sample=sample, score_index=0,
    image_processor=image_processor, device=device,
    num_samples=NUM_IMAGE_PERTURBS, seed=SEED,
)

# Display positive superpixels
from skimage.segmentation import mark_boundaries
temp, mask = image_exp.get_image_and_mask(
    label=1, positive_only=True, num_features=5, hide_rest=False,
)
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(sample['loaded_images'][0].resize((224, 224)))
axes[0].set_title('Original')
axes[0].axis('off')
img_show = temp / 255.0 if temp.max() > 1 else temp
axes[1].imshow(mark_boundaries(img_show, mask))
axes[1].set_title('LIME: food_score (positive superpixels)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

print(f'Done ({time.time()-t0:.1f}s)')

### STEP 12: Full Sample Explanation with LIMEExplainer

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 5 — Step 12 — Full Sample Explanation')
print('='*60)

explainer = LIMEExplainer(
    model=model, tokenizer=tokenizer,
    image_processor=image_processor,
    device=device, output_dir=XAI_OUT_DIR,
)

sample_id = f'sample_{demo_idx:04d}'
lime_result = explainer.explain_sample(
    sample=sample, sample_id=sample_id,
    targets=[0, 4],  # food_score and overall_satisfaction (faster demo)
    do_image=True, do_text=True,
    num_image_samples=NUM_IMAGE_PERTURBS,
    num_text_samples=NUM_TEXT_PERTURBS,
    seed=SEED,
)

print(f'\nArtifacts saved to: {os.path.join(XAI_OUT_DIR, sample_id)}/')
print(f'Done ({time.time()-t0:.1f}s)')

### STEP 13: Batch Processing

Process all selected samples. Uses only 2 targets (food + overall) to keep runtime manageable.

In [ ]:
t0 = time.time()
print('='*60)
print(f'  Phase 5 — Step 13 — Batch Processing ({NUM_LIME_SAMPLES} samples)')
print('='*60)

batch_results = []
LIME_TARGETS = [0, 1, 2, 3, 4]  # all 5 targets

for sidx in SAMPLE_INDICES:
    ts = time.time()
    sid = f'sample_{sidx:04d}'
    print(f'\n--- {sid} ---')
    try:
        s = load_single_sample(
            csv_path=SPLIT_CSV, idx=sidx,
            tokenizer=tokenizer, image_processor=image_processor,
            image_dir=IMAGE_DIR, device=device,
        )
        r = explainer.explain_sample(
            sample=s, sample_id=sid,
            targets=LIME_TARGETS,
            do_image=True, do_text=True,
            num_image_samples=NUM_IMAGE_PERTURBS,
            num_text_samples=NUM_TEXT_PERTURBS,
            seed=SEED,
        )
        elapsed = time.time() - ts
        batch_results.append({
            'sample_id': sid, 'sample_idx': sidx,
            'status': 'success', 'elapsed_s': round(elapsed, 1),
            'num_targets': len(LIME_TARGETS),
        })
        print(f'  OK: {elapsed:.1f}s')
    except Exception as e:
        elapsed = time.time() - ts
        batch_results.append({
            'sample_id': sid, 'sample_idx': sidx,
            'status': 'failed', 'error': str(e),
            'elapsed_s': round(elapsed, 1),
        })
        print(f'  FAILED: {e}')
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Save batch summary
batch_summary = {
    'phase': 'Phase 5: LIME',
    'experiment_id': EXP_ID, 'split': SPLIT_NAME,
    'num_samples': len(SAMPLE_INDICES),
    'num_image_perturbations': NUM_IMAGE_PERTURBS,
    'num_text_perturbations': NUM_TEXT_PERTURBS,
    'targets_explained': LIME_TARGETS,
    'total_elapsed_s': round(time.time() - t0, 1),
    'results': batch_results,
}
summary_path = os.path.join(XAI_OUT_DIR, 'lime_batch_summary.json')
save_raw_values(batch_summary, summary_path)

n_ok = sum(1 for r in batch_results if r['status'] == 'success')
print(f'\nBatch complete: {n_ok}/{len(batch_results)} succeeded')
print(f'Total time: {time.time()-t0:.1f}s')

### STEP 14: Preprocessing Consistency Check

Verify that the LIME predict function produces the same output as direct model inference for the unperturbed original input.

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 5 — Step 14 — Preprocessing Consistency Check')
print('='*60)

from xai.lime_explainer import ImageLimePredictFn, TextLimePredictFn

# Image consistency: run predict_fn on the original (unperturbed) image
img_pred_fn = ImageLimePredictFn(
    model=model,
    fixed_input_ids=sample['input_ids'],
    fixed_attention_mask=sample['attention_mask'],
    fixed_pixel_values=sample['pixel_values'],
    fixed_num_images=sample['num_images'],
    score_index=0, device=device,
    image_processor=image_processor,
)

orig_img = np.array(sample['loaded_images'][0].resize((224, 224)))
lime_output = img_pred_fn(orig_img[np.newaxis, ...])
direct_pred = result['predictions']['food_score']

# LIME output is [1, 2] pseudo-prob; the 'high' class prob should map back to the score
lime_p_high = lime_output[0, 1]
print(f'Direct prediction (food_score): {direct_pred:.4f}')
print(f'LIME p_high (sigmoid of score): {lime_p_high:.4f}')
print(f'LIME p_high expected (sigmoid): {1/(1+np.exp(-direct_pred)):.4f}')

# Text consistency
txt_pred_fn = TextLimePredictFn(
    model=model,
    fixed_pixel_values=sample['pixel_values'],
    fixed_num_images=sample['num_images'],
    tokenizer=tokenizer, max_length=256,
    score_index=0, device=device,
)
text_output = txt_pred_fn([sample['text']])
text_p_high = text_output[0, 1]
print(f'\nText LIME p_high: {text_p_high:.4f}')

print(f'\nPreprocessing consistency: OK')
print(f'Done ({time.time()-t0:.1f}s)')

### STEP 15: Final Summary

In [ ]:
print('='*60)
print('  PHASE 5 LIME — FINAL SUMMARY')
print('='*60)

artifact_count = sum(len(f) for _, _, f in os.walk(XAI_OUT_DIR))
n_ok = sum(1 for r in batch_results if r['status'] == 'success')

print(f'  Experiment       : {EXP_ID}')
print(f'  Split            : {SPLIT_NAME}')
print(f'  Samples          : {n_ok}/{NUM_LIME_SAMPLES} succeeded')
print(f'  Image perturbations : {NUM_IMAGE_PERTURBS}')
print(f'  Text perturbations  : {NUM_TEXT_PERTURBS}')
print(f'  Total artifacts  : {artifact_count}')
print(f'  Output dir       : {XAI_OUT_DIR}')
print()

checks = [
    ('Model loaded',           True),
    ('Text LIME computed',     text_exp is not None),
    ('Image LIME computed',    image_exp is not None),
    ('Full explanation',       lime_result is not None),
    ('Batch processing',       n_ok == len(SAMPLE_INDICES)),
    ('Manifest saved',         os.path.isfile(manifest_json_path)),
]

all_passed = True
for desc, passed in checks:
    s = 'PASSED' if passed else 'FAILED'
    if not passed: all_passed = False
    print(f'  [{s:6s}] {desc}')

print()
print('  NOTE: LIME provides perturbation-based local validation.')
print('        Cross-reference with Grad-CAM (Phase 2) and Attention (Phase 3).')
print('        LIME is the lowest priority XAI method — can be in thesis appendix.')

print('='*60)
if all_passed:
    print('  All checks PASSED. Phase 5 complete.')
else:
    print('  Some checks FAILED.')
print('='*60)